# EDA and Data Quality

Exploratory analysis of the Home Credit Default Risk dataset.  
Figures are saved to `reports/figures/` for inclusion in the LaTeX report.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Ensure credit_engine is importable from the notebooks/ directory
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = REPO_ROOT / 'dataset'

In [ ]:
from credit_engine.data_loader import load_data

df = load_data(DATA_DIR, mode='train')
print(f'Master join shape: {df.shape}')
print(f'Columns: {df.shape[1]} | Applicants: {df.shape[0]:,}')

In [ ]:
df.head()

---
## 1. Record Counts per Source Table After Master Join


In [ ]:
TABLE_COUNT_COLS = {
    'application (base)':    None,
    'bureau':                'bureau_cnt',
    'previous_application':  'prev_cnt',
    'pos_cash_balance':      'pos_cnt',
    'installments_payments': 'inst_cnt',
    'credit_card_balance':   'cc_cnt',
}
total_applicants = len(df)
rows = []
for table_name, cnt_col in TABLE_COUNT_COLS.items():
    if cnt_col is None:
        appl_w_rec = total_applicants
        tot_rec    = total_applicants
    else:
        appl_w_rec = int((df[cnt_col] > 0).sum())
        tot_rec    = int(df[cnt_col].sum())
    rows.append({'Source Table': table_name,
                 'Applicants with Records': appl_w_rec,
                 '% of Total': round(appl_w_rec / total_applicants * 100, 1),
                 'Total Secondary Records': tot_rec})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

secondary = summary[summary['Source Table'] != 'application (base)']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(secondary['Source Table'], secondary['Total Secondary Records'],
              color=sns.color_palette('tab10', n_colors=len(secondary)), edgecolor='white')
for bar, row in zip(bars, secondary.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + secondary['Total Secondary Records'].max() * 0.01,
            f'{row._4:,}\n({row._3}%)', ha='center', va='bottom', fontsize=8)
ax.set_title('Aggregated Record Counts per Secondary Source Table', fontsize=12)
ax.set_ylabel('Total Aggregated Records')
ax.tick_params(axis='x', rotation=20, labelsize=9)
ax.set_ylim(0, secondary['Total Secondary Records'].max() * 1.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_source_table_record_counts.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation.**

- Applicants with 0 secondary records receive zero-fills (count cols) and NaN (mean/max aggregates). Absence is itself a feature.
- `installments_payments` has the largest raw count — multiple payments per loan per month.

**→ `features.py` action:** Engineer `bureau_active_cnt / bureau_cnt` (active credit ratio), `prev_approved_cnt / prev_cnt` (approval rate), `inst_late_cnt / inst_cnt` (late payment rate) as normalised ratio features.

---
## 2. Target Variable — Loan Default Imbalance

The binary target `TARGET` encodes whether an applicant defaulted (1) or repaid (0).  
Understanding class imbalance is essential before any modelling decision.

In [ ]:
counts = df['TARGET'].value_counts().sort_index()
pcts   = (counts / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(7, 5))

bars = ax.bar(
    ['No Default (0)', 'Default (1)'],
    counts,
    color=['#2ecc71', '#e74c3c'],
    edgecolor='white',
    width=0.5,
)

for bar, cnt, pct in zip(bars, counts, pcts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + counts.max() * 0.01,
        f'{cnt:,}\n({pct}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold',
    )

ax.set_title('Target Variable Distribution: Loan Default Status', fontsize=13, pad=12)
ax.set_ylabel('Count', fontsize=11)
ax.set_xlabel('Loan Default Status', fontsize=11)
ax.set_ylim(0, counts.max() * 1.18)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_target_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation.**  
The dataset is heavily imbalanced: roughly **91–92% non-defaults** vs **8–9% defaults**.  
This has direct modelling consequences:

- A naive classifier predicting "no default" for every applicant achieves ~91% accuracy — accuracy is a misleading metric here.
- We will instead optimise on **Gini coefficient (= 2 × AUC − 1)** and **KS statistic**, both insensitive to class prevalence.
- Three imbalance-handling strategies will be benchmarked in `model.py`: **SMOTE** (applied only to training folds), **cost-sensitive learning** via `scale_pos_weight` / `is_unbalance`, and **threshold adjustment** on calibrated probabilities.
- Probability calibration (Platt scaling) is required: raw probabilities feed directly into Expected Loss = PD × LGD × EAD.

**→ `features.py` action:** No direct action, but class weights must be accounted for in any feature importance ranking — use LightGBM's `is_unbalance=True` during feature selection to avoid bias toward majority class.

---
## 3. Missing Value Heatmap — Top 40 Columns by Missingness Rate

Missing data patterns reveal both data quality issues and structural nulls.

In [ ]:
miss_rate = (df.isna().mean() * 100).sort_values(ascending=False)
top40_miss = miss_rate[miss_rate > 0].head(40)
miss_matrix = top40_miss.values.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(5, 14))
sns.heatmap(miss_matrix, ax=ax, annot=True, fmt='.1f', cmap='Reds',
            vmin=0, vmax=100, xticklabels=['Missing %'],
            yticklabels=top40_miss.index.tolist(), linewidths=0.3,
            cbar_kws={'label': 'Missing %'})
ax.set_title('Top 40 Columns by Missingness Rate (%)', fontsize=12, pad=10)
ax.tick_params(axis='y', labelsize=8)
ax.yaxis.tick_left()

plt.tight_layout()
plt.savefig(FIGURES_DIR / '03_missing_value_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Total columns with any missing values: {(miss_rate > 0).sum()}')
print(f'Columns with >50% missing: {(miss_rate > 50).sum()}')

**Interpretation.**

Two categories of missingness:
1. **Structural nulls** — `EXT_SOURCE_*` and secondary table aggregates: absence is informative (no bureau history = risk signal). Add `_missing_flag` indicators in `features.py`.
2. **Data quality gaps** — property columns (`APARTMENTS_AVG` etc.): missing for non-property owners. Drop or impute based on IV.

**→ `features.py` action:** Never impute before train/val split. Use LightGBM's native NaN handling for tree models. For Logistic Regression, median impute by category group.

---
## 4. Temporal Patterns in DAYS_ Columns

All `DAYS_*` columns encode time relative to application date as **negative integers**.  
Converted to positive years, they represent age, employment tenure, account age, etc.  
Non-linear patterns (U-shaped risk, plateau effects) motivate polynomial or bucketed features.

In [ ]:
# ── Unemployment sentinel ───────────────────────────────────────────────
UNEMPLOYMENT_SENTINEL = 365_243
n_unemployed = (df['DAYS_EMPLOYED'] == UNEMPLOYMENT_SENTINEL).sum()
pct_unemployed = n_unemployed / len(df) * 100
print(f'DAYS_EMPLOYED == {UNEMPLOYMENT_SENTINEL} (unemployed sentinel): '
      f'{n_unemployed:,} rows ({pct_unemployed:.1f}%)')
print(f'Default rate for unemployed: '
      f'{df[df["DAYS_EMPLOYED"] == UNEMPLOYMENT_SENTINEL]["TARGET"].mean():.4f}')
print(f'Default rate for employed:   '
      f'{df[df["DAYS_EMPLOYED"] != UNEMPLOYMENT_SENTINEL]["TARGET"].mean():.4f}')

# ── KDE plots for 3 temporal features ──────────────────────────────────
TEMPORAL_COLS = ['DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH']
TEMPORAL_LABELS = {
    'DAYS_EMPLOYED':    'Employment Tenure (years)',
    'DAYS_REGISTRATION': 'Years Since Address Registration',
    'DAYS_ID_PUBLISH':  'Years Since ID Document Issued',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {0: '#2ecc71', 1: '#e74c3c'}

for ax, col in zip(axes, TEMPORAL_COLS):
    for t, colour in palette.items():
        # Remove unemployment sentinel (365243) and negatives → convert to years
        subset = df[(df['TARGET'] == t) &
                    (df[col] < 0) &
                    (df[col] != -UNEMPLOYMENT_SENTINEL)][col]
        years = (-subset / 365.25).clip(upper=subset.abs().quantile(0.99) / 365.25)
        if len(years) > 200:
            sns.kdeplot(years, ax=ax, color=colour, fill=True, alpha=0.35,
                        linewidth=1.5, label='Default' if t == 1 else 'No Default')
    ax.set_xlabel('Years', fontsize=9)
    ax.set_title(TEMPORAL_LABELS[col], fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Temporal Feature KDE: Default vs Non-Default (employment sentinel excluded)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_temporal_patterns_kde.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Default rate by employment tenure bins ──────────────────────────────
df_emp = df[df['DAYS_EMPLOYED'] < 0].copy()
df_emp['emp_years'] = -df_emp['DAYS_EMPLOYED'] / 365.25
df_emp['emp_bin'] = pd.cut(df_emp['emp_years'],
                            bins=[0, 1, 2, 5, 10, 20, 50],
                            labels=['<1yr', '1-2yr', '2-5yr', '5-10yr', '10-20yr', '>20yr'])
emp_dr = df_emp.groupby('emp_bin', observed=True)['TARGET'].agg(['mean', 'count'])
print('\nDefault rate by employment tenure:')
print(emp_dr.rename(columns={'mean': 'default_rate', 'count': 'n'}))

**Interpretation.**

- **Unemployment sentinel (365,243)**: These applicants have a materially *higher* default rate than employed applicants. They must not be included in continuous feature distributions — they need a separate binary flag.
- **Employment tenure**: Default rate is typically highest for very new employees (< 1 year) and declines monotonically — consistent with job stability as a risk indicator. This suggests a **non-linear transformation** or **bucketing** will outperform a linear feature.
- **DAYS_REGISTRATION / DAYS_ID_PUBLISH**: Recent registrations (< 1 year) often have higher default rates, possibly indicating address instability or recently issued documents under a different identity.

**→ `features.py` actions:**
- Create `app_is_unemployed = (DAYS_EMPLOYED == 365243).astype(int)` — binary flag.
- For employed applicants: `app_employment_years = -DAYS_EMPLOYED / 365.25` (clip sentinel first).
- Create employment tenure bins matching the table above: `app_employment_bin` (5–6 buckets).
- Add `app_id_age_years = -DAYS_ID_PUBLISH / 365.25` and `app_registration_years = -DAYS_REGISTRATION / 365.25`.
- Consider `app_employment_years_sq` (polynomial term) if tenure default rate is clearly non-linear.

---
## 5. Feature Distributions — Default vs Non-Default

Overlaid distributions reveal whether individual features separate the two classes.  
Strong separation → high Information Value → strong WoE binning candidate.

In [ ]:
FEATURES = [
    'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'DAYS_EMPLOYED',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_ANNUITY', 'DAYS_ID_PUBLISH', 'AMT_GOODS_PRICE',
]
LABELS = {
    'AMT_CREDIT':       'Credit Amount (USD)',
    'AMT_INCOME_TOTAL': 'Annual Income (USD)',
    'DAYS_BIRTH':       'Age (years)',
    'DAYS_EMPLOYED':    'Employment Length (years)',
    'EXT_SOURCE_1':     'Ext. Score 1',
    'EXT_SOURCE_2':     'Ext. Score 2',
    'EXT_SOURCE_3':     'Ext. Score 3',
    'AMT_ANNUITY':      'Annuity Amount (USD)',
    'DAYS_ID_PUBLISH':  'ID Document Age (years)',
    'AMT_GOODS_PRICE':  'Goods Price (USD)',
}

def _to_plot_series(df, col):
    s = df[col].dropna()
    if col in ('DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH'):
        s = s.clip(upper=0)
        s = (-s / 365.25)
    return s

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
palette = {0: '#2ecc71', 1: '#e74c3c'}

for ax, col in zip(axes, FEATURES):
    for target_val, colour in palette.items():
        data = _to_plot_series(df[df['TARGET'] == target_val], col)
        if data.empty:
            continue
        sns.histplot(data, ax=ax, kde=True, color=colour,
                     label='Default' if target_val == 1 else 'No Default',
                     stat='density', alpha=0.45, linewidth=0, bins=40)
    ax.set_title(LABELS[col], fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

fig.suptitle('Feature Distributions: Default (red) vs Non-Default (green)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '05_feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation.**

| Feature | Observation | `features.py` action |
|---|---|---|
| **EXT_SOURCE_2/3** | Strongest separation — defaulters cluster at lower scores | Retain raw + add missingness flag |
| **DAYS_BIRTH (Age)** | Younger applicants default at higher rates | Convert to years; consider polynomial |
| **DAYS_EMPLOYED** | 365,243 sentinel for unemployment; short tenure → higher default | Clip + binary `is_unemployed` flag |
| **AMT_INCOME_TOTAL** | Heavy right skew | Log-transform in `features.py` |
| **AMT_CREDIT / AMT_GOODS_PRICE** | Moderate overlap; multimodal | Engineer LTV = AMT_CREDIT / AMT_GOODS_PRICE |
| **EXT_SOURCE_1** | ~44% missing | Median impute by education group + `ext1_missing_flag` |

---
## 6. Correlation Heatmap — Top 30 Features by Variance


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in {'SK_ID_CURR', 'TARGET'}]
top30_cols = df[numeric_cols].var().nlargest(30).index.tolist()
corr = df[top30_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, ax=ax, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', square=True, linewidths=0.3,
            annot_kws={'size': 6}, cbar_kws={'label': 'Pearson r', 'shrink': 0.8})
ax.set_title('Pearson Correlation — Top 30 Features by Variance', fontsize=13, pad=12)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '06_correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation.**

- `AMT_CREDIT` ↔ `AMT_GOODS_PRICE` (~0.98): replace both with **LTV ratio** in `features.py`.
- `AMT_ANNUITY` ↔ `AMT_CREDIT` (high): replace with **debt-service ratio** = AMT_ANNUITY / AMT_INCOME_TOTAL.
- `inst_amt_payment_sum` ↔ `inst_cnt`: use mean/ratio not raw sums.
- `EXT_SOURCE_1/2/3`: low mutual correlation — retain all three independently.

**→ `features.py` action:** For Logistic Regression, drop any feature with |r| > 0.85 with another retained feature.

---
## 7. Categorical Default Rates & Weight of Evidence (WoE)

**WoE** = ln(% non-defaults in bin / % defaults in bin).  
**Information Value (IV)** = Σ (% non-defaults − % defaults) × WoE across all bins.  
IV > 0.3 = strong predictor; IV < 0.02 = useless.  
WoE-encoded features are required for the Logistic Regression benchmark (linearity assumption satisfied) and are accepted under Basel IRB frameworks.

In [ ]:
CATEGORICAL_COLS = [
    'CODE_GENDER',
    'NAME_EDUCATION_TYPE',
    'NAME_INCOME_TYPE',
    'NAME_CONTRACT_TYPE',
    'OCCUPATION_TYPE',
]

total_defaults     = df['TARGET'].sum()
total_non_defaults = len(df) - total_defaults
iv_summary = {}

fig, axes = plt.subplots(len(CATEGORICAL_COLS), 2,
                          figsize=(14, 4 * len(CATEGORICAL_COLS)))

for row_idx, col in enumerate(CATEGORICAL_COLS):
    grp = (
        df.groupby(col, observed=True)['TARGET']
          .agg(defaults='sum', total='count')
          .assign(non_defaults=lambda x: x['total'] - x['defaults'])
          .assign(default_rate=lambda x: x['defaults'] / x['total'])
    )
    # Avoid log(0) — clip small counts
    EPSILON = 0.5
    grp['pct_def']     = (grp['defaults']     + EPSILON) / (total_defaults     + EPSILON * len(grp))
    grp['pct_non_def'] = (grp['non_defaults'] + EPSILON) / (total_non_defaults + EPSILON * len(grp))
    grp['woe']         = np.log(grp['pct_non_def'] / grp['pct_def'])
    grp['iv_part']     = (grp['pct_non_def'] - grp['pct_def']) * grp['woe']
    iv_summary[col]    = grp['iv_part'].sum()

    grp = grp.sort_values('default_rate', ascending=True)
    ax_rate, ax_woe = axes[row_idx]

    # Default rate bar
    grp['default_rate'].plot(kind='barh', ax=ax_rate, color='#e74c3c', alpha=0.75)
    ax_rate.axvline(df['TARGET'].mean(), color='black', linestyle='--', linewidth=1,
                    label=f'Overall rate ({df["TARGET"].mean():.3f})')
    ax_rate.set_title(f'{col} — Default Rate', fontsize=9, fontweight='bold')
    ax_rate.set_xlabel('Default Rate')
    ax_rate.legend(fontsize=7)

    # WoE bar
    colours_woe = ['#2ecc71' if v > 0 else '#e74c3c' for v in grp['woe']]
    grp['woe'].plot(kind='barh', ax=ax_woe, color=colours_woe, alpha=0.8)
    ax_woe.axvline(0, color='black', linewidth=1)
    ax_woe.set_title(f'{col} — WoE  (IV={iv_summary[col]:.3f})', fontsize=9, fontweight='bold')
    ax_woe.set_xlabel('Weight of Evidence')

plt.suptitle('Categorical Features: Default Rate and Weight of Evidence per Group',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_categorical_woe.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nInformation Value Summary:')
for col, iv in sorted(iv_summary.items(), key=lambda x: -x[1]):
    strength = ('Strong' if iv > 0.3 else
                'Medium' if iv > 0.1 else
                'Weak'   if iv > 0.02 else 'Useless')
    print(f'  {col:<30s}  IV={iv:.4f}  ({strength})')

**Interpretation.**

WoE bars crossing zero separate low-risk groups (positive WoE = more non-defaults than average) from high-risk groups (negative WoE = more defaults than average).  
IV measures the total predictive power of the entire feature.

Typical findings for Home Credit:
- **NAME_EDUCATION_TYPE**: Higher education → lower WoE (lower risk). IV usually ~0.08–0.15 (Medium).
- **NAME_INCOME_TYPE**: Working class vs pensioners show distinct risk profiles. IV ~0.05–0.12.
- **OCCUPATION_TYPE**: Large variance across occupations; high-risk occupations (labourers) vs low-risk (managers). IV often > 0.1.
- **CODE_GENDER**: Female applicants historically lower default rate in this dataset. IV ~0.01–0.05.
- **NAME_CONTRACT_TYPE**: Cash loans vs revolving loans may differ. IV ~0.01–0.03.

**→ `features.py` actions:**
- For **Logistic Regression**: replace each categorical with its WoE value (map each category → WoE scalar computed on training fold only — never on full dataset to avoid leakage).
- For **LightGBM/XGBoost**: ordinal-encode or leave as category dtype (these models handle categoricals natively).
- Collapse groups with |WoE| < 0.05 into a single 'Other' bin to avoid overfitting on rare categories.
- Add `OCCUPATION_TYPE_woe`, `NAME_EDUCATION_TYPE_woe`, `NAME_INCOME_TYPE_woe` as engineered numeric features.

---
## 8. Loan-to-Value & Debt-Service Ratio Analysis

Raw monetary features (`AMT_CREDIT`, `AMT_ANNUITY`, `AMT_INCOME_TOTAL`) show weak univariate separation.  
**Ratios** derived from them capture the financial affordability concept that directly drives default:  
a high loan relative to income, or a high monthly repayment relative to income, are classic Basel PD inputs.

In [ ]:
# Compute derived financial ratios — always guard against division by zero
df_r = df[['TARGET', 'AMT_CREDIT', 'AMT_GOODS_PRICE',
           'AMT_ANNUITY', 'AMT_INCOME_TOTAL']].copy()

df_r['ltv'] = np.where(
    df_r['AMT_GOODS_PRICE'] > 0,
    df_r['AMT_CREDIT'] / df_r['AMT_GOODS_PRICE'], np.nan)

df_r['debt_service_ratio'] = np.where(
    df_r['AMT_INCOME_TOTAL'] > 0,
    df_r['AMT_ANNUITY'] / df_r['AMT_INCOME_TOTAL'], np.nan)

df_r['income_to_credit'] = np.where(
    df_r['AMT_CREDIT'] > 0,
    df_r['AMT_INCOME_TOTAL'] / df_r['AMT_CREDIT'], np.nan)

RATIO_LABELS = {
    'ltv':               'LTV  (AMT_CREDIT / AMT_GOODS_PRICE)',
    'debt_service_ratio': 'Debt-Service Ratio  (Annuity / Income)',
    'income_to_credit':  'Income-to-Credit  (Income / Credit)',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {0: '#2ecc71', 1: '#e74c3c'}

for ax, col in zip(axes, ['ltv', 'debt_service_ratio', 'income_to_credit']):
    for t, colour in palette.items():
        data = df_r[df_r['TARGET'] == t][col].dropna()
        # Clip extreme outliers for display (99th percentile)
        clip_hi = data.quantile(0.99)
        data = data.clip(upper=clip_hi)
        sns.kdeplot(data, ax=ax, color=colour,
                    label='Default' if t == 1 else 'No Default',
                    fill=True, alpha=0.35, linewidth=1.5)
    ax.set_title(RATIO_LABELS[col], fontsize=9, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Derived Financial Ratio Distributions by Default Status', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_financial_ratios.png', dpi=100, bbox_inches='tight')
plt.show()

# Summary statistics per class
for col in ['ltv', 'debt_service_ratio', 'income_to_credit']:
    stats = df_r.groupby('TARGET')[col].median().rename(index={0:'Non-default', 1:'Default'})
    print(f'Median {col}:  {stats.to_dict()}')

**Interpretation.**

| Ratio | Expected finding | Why it matters |
|---|---|---|
| **LTV** (credit / goods price) | Defaulters typically > 1.0 — borrowing *above* the goods value | Signals top-up loan or interest capitalisation; strong default indicator |
| **Debt-service ratio** (annuity / income) | Defaulters have higher DSR — spending larger share of income on repayment | Basel II affordability metric; regulatory PD input |
| **Income-to-credit** | Defaulters have lower ratio — low income relative to loan size | Inverse of leverage; lower = more stressed |

The KDE separation between classes on these ratios is typically *stronger* than on the raw components, confirming the interaction effect.

**→ `features.py` actions:**
- Engineer `app_ltv = AMT_CREDIT / AMT_GOODS_PRICE` — replace raw pair
- Engineer `app_debt_service_ratio = AMT_ANNUITY / AMT_INCOME_TOTAL`
- Engineer `app_income_to_credit = AMT_INCOME_TOTAL / AMT_CREDIT`
- Log-transform `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_GOODS_PRICE` before use in linear models
- Cap LTV at 99th percentile to handle data entry errors (LTV > 5 is implausible)

---
## 9. Secondary Table Richness & Absence Effects

Applicants with **no secondary history** (bureau_cnt = 0, prev_cnt = 0, etc.) are so-called *thin-file* borrowers.  
The combination of which secondary tables a borrower appears in — and which they don't — is itself a risk signal.

In [ ]:
# Binary presence/absence flags
df_cov = df[['TARGET', 'bureau_cnt', 'prev_cnt', 'inst_cnt', 'cc_cnt']].copy()
df_cov['has_bureau'] = (df_cov['bureau_cnt'] > 0).astype(int)
df_cov['has_prev']   = (df_cov['prev_cnt']   > 0).astype(int)
df_cov['has_inst']   = (df_cov['inst_cnt']   > 0).astype(int)
df_cov['has_cc']     = (df_cov['cc_cnt']     > 0).astype(int)

COVERAGE_COLS = ['has_bureau', 'has_prev', 'has_inst', 'has_cc']

# ── Default rate per flag ───────────────────────────────────────────────
print('Default rate by secondary table presence (0=absent, 1=present):\n')
for col in COVERAGE_COLS:
    dr = df_cov.groupby(col)['TARGET'].agg(['mean', 'count'])
    print(f'  {col}:')
    for idx, row in dr.iterrows():
        print(f'    {idx}  →  default_rate={row["mean"]:.4f}  (n={row["count"]:,})')

# ── 2×2 interaction heatmaps ────────────────────────────────────────────
from itertools import combinations
pairs = list(combinations(COVERAGE_COLS, 2))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, (c1, c2) in zip(axes, pairs):
    pivot = df_cov.pivot_table(values='TARGET', index=c1, columns=c2, aggfunc='mean')
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn_r',
                vmin=0.04, vmax=0.16, cbar_kws={'label': 'Default Rate'})
    ax.set_title(f'{c1.replace("has_","")} × {c2.replace("has_","")}', fontsize=9, fontweight='bold')
    ax.set_xlabel(c2.replace('has_', 'has '))
    ax.set_ylabel(c1.replace('has_', 'has '))

fig.suptitle('Default Rate by Pairwise Secondary Table Presence (0=absent, 1=present)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '09_secondary_table_coverage_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Thin-credit definition ──────────────────────────────────────────────
df_cov['thin_credit'] = ((df_cov['has_bureau'] == 0) &
                          (df_cov['has_prev']   == 0)).astype(int)
thin_stats = df_cov.groupby('thin_credit')['TARGET'].agg(['mean', 'count'])
print('\nThin-credit (no bureau AND no previous application):')
print(thin_stats.rename(columns={'mean': 'default_rate', 'count': 'n'}))

**Interpretation.**

The heatmap cells reveal interaction effects:
- **No bureau AND no prev** (bottom-left cell of bureau×prev): typically the *highest* default rate — true thin-file borrowers with no credit history at all.
- **Has bureau but no installments**: older credit history without recent repayment activity — moderate risk.
- **Has all four**: richest file; typically lowest default rate as lender has most information.

The *combination* of table absence is more informative than any single flag — this is a known credit bureau phenomenon where thin-file applicants are systematically under-scored.

**→ `features.py` actions:**
- Add binary flags: `bureau_history_flag`, `prev_history_flag`, `inst_history_flag`, `cc_history_flag`.
- Add composite: `app_thin_credit_flag = (bureau_cnt == 0) & (prev_cnt == 0)`.
- Add credit richness score: `app_data_richness = has_bureau + has_prev + has_inst + has_cc` (0–4 integer).
- **Do NOT impute NaN secondary aggregate columns with median** for thin-file applicants — their nulls are structural and should be kept as NaN (LightGBM handles them) or filled with a sentinel (e.g., −1 for Logistic Regression).

---
## 10. Secondary Table Aggregate — Predictive Power Ranking

The 5 secondary tables produced 30+ aggregate columns.  
Not all are equally predictive. Point-biserial correlation with TARGET identifies which aggregates carry signal
and therefore warrant further ratio/flag engineering — vs which can be dropped to reduce dimensionality.

In [ ]:
SECONDARY_PREFIXES = ('bureau_', 'prev_', 'pos_', 'inst_', 'cc_')
agg_cols = [c for c in df.columns
            if any(c.startswith(p) for p in SECONDARY_PREFIXES)]

correlations = {}
for col in agg_cols:
    valid = df[[col, 'TARGET']].dropna()
    if len(valid) > 100:
        correlations[col] = valid[col].corr(valid['TARGET'])

corr_series = (pd.Series(correlations)
               .sort_values(key=abs, ascending=False))

# Plot top 20 by absolute correlation
top20 = corr_series.head(20)
colours = ['#e74c3c' if v > 0 else '#2ecc71' for v in top20.values]

fig, ax = plt.subplots(figsize=(10, 8))
top20.sort_values().plot(kind='barh', ax=ax, color=colours[::-1], alpha=0.85)
ax.axvline(0.05,  color='orange', linestyle='--', linewidth=1, label='Weak signal |r|=0.05')
ax.axvline(-0.05, color='orange', linestyle='--', linewidth=1)
ax.axvline(0.15,  color='red',    linestyle='--', linewidth=1, label='Strong signal |r|=0.15')
ax.axvline(-0.15, color='red',    linestyle='--', linewidth=1)
ax.set_title('Top 20 Secondary Aggregates: Point-Biserial Correlation with TARGET',
             fontsize=11, pad=10)
ax.set_xlabel('Pearson r with TARGET (positive = more defaults)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '10_secondary_table_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nAll secondary aggregate correlations (|r| > 0.05):')
sig = corr_series[corr_series.abs() > 0.05]
for name, val in sig.items():
    print(f'  {name:<45s}  r={val:+.4f}')

print(f'\nTotal secondary aggregates: {len(correlations)}')
print(f'With |r| > 0.05 (weak+): {len(sig)}')
print(f'With |r| > 0.15 (strong): {(corr_series.abs() > 0.15).sum()}')

**Interpretation.**

Expected top-ranking secondary aggregates:
- **`inst_days_past_due_mean`** / **`inst_late_cnt`** — payment behaviour is a direct default predictor (positive r: more late payments → more defaults).
- **`bureau_overdue_max`** — maximum overdue days across all bureau entries; strong positive correlation.
- **`bureau_bbal_dpd_rate_mean`** — mean DPD rate from bureau balance history.
- **`cc_sk_dpd_max`** — maximum credit card days past due.
- **`prev_refused_cnt`** — prior application refusals are a signal of previous rejection history.

Low-signal aggregates (|r| < 0.05) likely include: `pos_months_balance_mean`, `cc_drawing_mean`, `bureau_prolong_sum`. These do not warrant further derived features.

**→ `features.py` actions:**
- For high-signal count aggregates (|r| > 0.05 on raw count), engineer **normalised ratios**: `inst_late_rate = inst_late_cnt / (inst_cnt + 1)`, `prev_refused_rate = prev_refused_cnt / (prev_cnt + 1)`, `bureau_active_rate = bureau_active_cnt / (bureau_cnt + 1)`.
- For DPD aggregates with strong signal, also create **binary flags**: `bureau_ever_overdue_flag = (bureau_overdue_max > 0).astype(int)`.
- Drop aggregates with |r| < 0.02 from the Logistic Regression feature set (keep for tree models which can handle noise).

---
## 11. Payment Behaviour Trends

Late payment *rate* (normalised) and payment *shortfall* are stronger default signals than raw late counts  
because they account for varying instalment history length.  
This section validates whether these derived signals separate default classes before engineering them.

In [ ]:
df_pay = df[['TARGET', 'inst_cnt', 'inst_late_cnt',
             'inst_payment_ratio_mean', 'inst_days_past_due_mean',
             'cc_sk_dpd_max', 'bureau_overdue_max']].copy()

# Derived features
df_pay['inst_late_rate'] = np.where(
    df_pay['inst_cnt'] > 0,
    df_pay['inst_late_cnt'] / df_pay['inst_cnt'], np.nan)

# Underpayment flag: paid less than 95% of instalment on average AND has late payments
df_pay['inst_underpayment_flag'] = (
    (df_pay['inst_payment_ratio_mean'].fillna(1.0) < 0.95) &
    (df_pay['inst_late_cnt'] > 0)
).astype(int)

# ── Default rate by underpayment flag ───────────────────────────────────
print('Default rate by inst_underpayment_flag:')
print(df_pay.groupby('inst_underpayment_flag')['TARGET']
      .agg(['mean', 'count'])
      .rename(columns={'mean': 'default_rate', 'count': 'n'}))

# ── Distribution plots for key payment features ─────────────────────────
PAYMENT_FEATURES = [
    ('inst_late_rate',           'Late Payment Rate (inst_late / inst_cnt)'),
    ('inst_payment_ratio_mean',  'Mean Payment Ratio (paid / scheduled)'),
    ('inst_days_past_due_mean',  'Mean Days Past Due (instalments)'),
    ('bureau_overdue_max',       'Max Bureau Overdue Days'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
palette = {0: '#2ecc71', 1: '#e74c3c'}

for ax, (col, label) in zip(axes, PAYMENT_FEATURES):
    for t, colour in palette.items():
        data = df_pay[df_pay['TARGET'] == t][col].dropna()
        # Clip at 99th percentile for display
        data = data.clip(upper=data.quantile(0.99))
        if len(data) > 200:
            sns.kdeplot(data, ax=ax, color=colour, fill=True, alpha=0.35,
                        linewidth=1.5, label='Default' if t == 1 else 'No Default')
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Payment Behaviour Features: Default vs Non-Default Distributions',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '11_payment_behaviour.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Median comparison by class ───────────────────────────────────────────
print('\nMedian payment behaviour by TARGET:')
cols_to_compare = ['inst_late_rate', 'inst_payment_ratio_mean',
                   'inst_days_past_due_mean', 'bureau_overdue_max']
print(df_pay.groupby('TARGET')[cols_to_compare].median().T.rename(
    columns={0: 'Non-default', 1: 'Default'}).round(4))

**Interpretation.**

| Feature | Expected finding | Strength |
|---|---|---|
| **`inst_late_rate`** | Defaulters have substantially higher late payment rate | Strong |
| **`inst_payment_ratio_mean`** | Defaulters pay a lower fraction of scheduled amount | Moderate |
| **`inst_days_past_due_mean`** | Defaulters have more DPD on average, but distributions overlap (many zero-DPD defaulters) | Moderate |
| **`bureau_overdue_max`** | Strong right tail for defaulters — a single severe overdue event is highly predictive | Strong |

The `inst_underpayment_flag` default rate comparison quantifies the combined signal: applicants who both underpay and pay late should have a materially higher default rate than the overall 8–9%.

**→ `features.py` actions:**
- Engineer `inst_late_rate = inst_late_cnt / (inst_cnt + 1)` — the +1 avoids division by zero for applicants with exactly 1 instalment.
- Engineer `inst_payment_shortfall = 1 - inst_payment_ratio_mean` (closer to 0 = better; positive = underpaid).
- Create `bureau_ever_overdue_flag = (bureau_overdue_max > 0).astype(int)`.
- Create `cc_ever_dpd_flag = (cc_sk_dpd_max > 0).astype(int)` (card-level DPD).
- Create `inst_underpayment_flag` as defined above — binary interaction of underpayment + late.
- **Do not** create a recency-weighted trend feature from these aggregates alone — the raw tables would be needed for that; the existing aggregates are already time-collapsed.

---

---
## 12. Age × Income Interaction — 2D Default Rate Heatmap

Individual univariate distributions showed that both age and income associate with default.  
This analysis tests whether the combination (young **and** low income) creates a multiplicative risk  
beyond what either variable contributes independently.

In [ ]:
df_ai = df[['TARGET', 'DAYS_BIRTH', 'AMT_INCOME_TOTAL']].copy()
df_ai['age'] = (-df_ai['DAYS_BIRTH'] / 365.25).clip(lower=18, upper=80)

# Remove extreme income outliers before binning
income_cap = df_ai['AMT_INCOME_TOTAL'].quantile(0.99)
df_ai = df_ai[df_ai['AMT_INCOME_TOTAL'] <= income_cap].copy()

df_ai['age_bin'] = pd.cut(
    df_ai['age'],
    bins=[18, 25, 30, 35, 40, 50, 65, 80],
    labels=['18–25', '25–30', '30–35', '35–40', '40–50', '50–65', '65–80'],
    right=True,
)
df_ai['income_quintile'] = pd.qcut(
    df_ai['AMT_INCOME_TOTAL'], q=5,
    labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)'],
    duplicates='drop',
)

pivot_dr    = df_ai.pivot_table(values='TARGET', index='age_bin',
                                 columns='income_quintile', aggfunc='mean')
pivot_count = df_ai.pivot_table(values='TARGET', index='age_bin',
                                 columns='income_quintile', aggfunc='count')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(pivot_dr, ax=axes[0], annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0.04, vmax=0.18, cbar_kws={'label': 'Default Rate'},
            linewidths=0.3)
axes[0].set_title('Default Rate: Age Bin × Income Quintile', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Income Quintile')
axes[0].set_ylabel('Age Group')

sns.heatmap(pivot_count, ax=axes[1], annot=True, fmt=',', cmap='Blues',
            cbar_kws={'label': 'Count'}, linewidths=0.3)
axes[1].set_title('Cell Counts: Age Bin × Income Quintile', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Income Quintile')
axes[1].set_ylabel('Age Group')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '12_age_income_interaction_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

# Marginal effects for comparison
print('Marginal default rate by age bin:')
print(df_ai.groupby('age_bin', observed=True)['TARGET'].mean().round(4).to_string())
print('\nMarginal default rate by income quintile:')
print(df_ai.groupby('income_quintile', observed=True)['TARGET'].mean().round(4).to_string())

**Interpretation.**

Reading the heatmap:
- **Top-left cell (18–25 × Q1 lowest income)**: typically the highest default rate cell in the entire matrix — confirming the interaction hypothesis. Young + low income is multiplicatively riskier than either factor alone.
- **Bottom-right (65–80 × Q5 highest income)**: typically the lowest default rate — experienced borrower with high income.
- **Income gradient**: Default rate usually decreases monotonically from Q1 to Q5 within each age group — income is protective regardless of age.
- **Age gradient**: Weakest for high-income applicants (Q5) — wealthy young applicants default at similar rates to wealthy older ones.

The cell counts heatmap reveals which cells are sparse (< 500 rows) — default rate estimates there are unreliable and should not drive binning decisions.

**→ `features.py` actions:**
- Add **age × income interaction**: `app_age_income_interaction = app_age_years * np.log1p(AMT_INCOME_TOTAL)` — the log-transform prevents income scale dominance.
- Add **age buckets** as a categorical feature: `app_age_group` with the bins shown (18–25, 25–30, ..., 65+).
- For Logistic Regression, WoE-encode `app_age_group × income_quintile` combination if the interaction is strong enough (IV > 0.1).
- Verify: run a quick LightGBM with and without the interaction feature to confirm SHAP contribution before committing.

---
## EDA Complete — Feature Engineering Checklist for `features.py`

| Category | Features to engineer | Source analysis |
|---|---|---|
| **Financial ratios** | `app_ltv`, `app_debt_service_ratio`, `app_income_to_credit` | §6 |
| **Log transforms** | `log_amt_income`, `log_amt_credit`, `log_amt_goods_price` | §2, §6 |
| **WoE encoding** | `occ_type_woe`, `edu_type_woe`, `income_type_woe` | §7 |
| **Employment** | `app_is_unemployed`, `app_employment_years`, `app_employment_bin` | §9 |
| **Temporal** | `app_age_years`, `app_id_age_years`, `app_registration_years` | §2, §9 |
| **Normalised ratios** | `inst_late_rate`, `prev_refused_rate`, `bureau_active_rate` | §8 |
| **DPD flags** | `bureau_ever_overdue_flag`, `cc_ever_dpd_flag`, `inst_underpayment_flag` | §8, §12 |
| **Thin-credit** | `bureau_history_flag`, `prev_history_flag`, `app_thin_credit_flag`, `app_data_richness` | §10 |
| **Interaction** | `app_age_income_interaction` | §11 |
| **Missingness flags** | `ext1_missing_flag`, `ext2_missing_flag`, `ext3_missing_flag` | §3 |